# Lecture 8.3: Custom Tools with Real Data

**Section 8: MCP, Connecting External Systems**

In the last lecture you wrapped a pure function as a tool. The tool took a number and returned a number, and nothing else in the world changed.

Real tools are different. They read your application's data, and some of them change it. In this notebook you build three tools for **Pixel Forge**, a two-person indie game studio, so an agent can read and update the studio's sprint board.

| Tool | What it does | Changes the board? |
|---|---|---|
| `list_tasks` | Lists tasks, optionally for one column | No (read-only) |
| `add_task` | Adds a new task to `todo` | Yes |
| `move_task` | Moves a task to another column | Yes |

Along the way you will see:

- **Full JSON Schema** for enums and required fields
- **Optional arguments** read with `args.get()`
- **Validation** that rejects bad input before your function runs
- **`is_error`** results that the agent can act on
- **`readOnlyHint`**, which lets Claude call safe tools side by side

Every change is saved to `/content/pixelforge/board.json`. That file on disk is the ground truth. The agent's wording will vary from run to run. The file will not lie.

> **Code-along version.** Cells 7 and 8 are empty. Type them as you follow the video, then run the notebook from the top.

## Cell 1: Install the Claude Agent SDK

This cell installs the Claude Agent SDK, the only package this notebook needs. The SDK also brings in the `mcp` package, which provides the in-process MCP server your custom tools run on.

The version is **pinned** so every example behaves the way it does in the video. The comment in the cell shows how to install the latest version instead. Newer versions may behave differently.

If the package is already installed in this Colab session, pip notices and finishes quickly.

In [ ]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.188 -q

## Cell 2: Load your API key from Colab Secrets

The agent needs your Anthropic API key. This notebook reads it from **Colab Secrets**, so the key never appears in the notebook itself.

**Add the secret once:**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key into the value field. You create keys in the Claude Console at https://platform.claude.com/ under **API Keys**.
5. Switch **Notebook access** on.

The cell copies the secret into the `ANTHROPIC_API_KEY` environment variable, which is where the SDK looks for it.

> **Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before you start Jupyter.

In [ ]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` holds the model every agent in this notebook uses. It is passed as `model=MODEL_NAME` in the options, so this is the **one place** to change the model for the whole notebook.

The default is Claude Haiku 5.5. Model names change with new releases, so check the models page linked in the comment for the latest list.

In [ ]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Meet the Pixel Forge sprint board

Pixel Forge has two developers, **Maya** and **Arjun**. They track their work on a sprint board with three columns: `todo`, `doing` and `done`.

The board is an ordinary Python dictionary called `BOARD`. Each key is a task ID, and each value holds the task's title, column and owner.

| Helper | What it does |
|---|---|
| `save_board()` | Writes `BOARD` to `/content/pixelforge/board.json`. The tools call it after every change. |
| `show_board()` | Prints the board grouped by column. |
| `read_board_file()` | Reads `board.json` back from disk. The verification cells use it as ground truth. |

The board starts with 8 tasks: 3 in `todo`, 3 in `doing` and 2 in `done`. Running this cell again resets the board to that starting state.

**Why a file on disk?** The agent will describe what it did in its own words, and those words vary. The file records what actually happened. Every check later in this notebook reads it.

In [ ]:
import json
from pathlib import Path

BOARD_PATH = Path("/content/pixelforge/board.json")
BOARD_PATH.parent.mkdir(parents=True, exist_ok=True)

COLUMNS = ["todo", "doing", "done"]

BOARD = {
    "T-1": {"title": "Fix jump physics", "column": "doing", "owner": "Arjun"},
    "T-2": {"title": "Draw forest tileset", "column": "doing", "owner": "Maya"},
    "T-3": {"title": "Balance boss fight", "column": "doing", "owner": "Arjun"},
    "T-4": {"title": "Compose title theme", "column": "todo", "owner": "Maya"},
    "T-5": {"title": "Add save slots", "column": "todo", "owner": "Arjun"},
    "T-6": {"title": "Design potion icons", "column": "todo", "owner": "Maya"},
    "T-7": {"title": "Set up build pipeline", "column": "done", "owner": "Arjun"},
    "T-8": {"title": "Write tutorial text", "column": "done", "owner": "Maya"},
}


def save_board():
    BOARD_PATH.write_text(json.dumps(BOARD, indent=2))


def show_board():
    for column in COLUMNS:
        print(column.upper())
        for task_id, task in BOARD.items():
            if task["column"] == column:
                print(f"  {task_id}  {task['title']}  ({task['owner']})")


def read_board_file():
    return json.loads(BOARD_PATH.read_text())


save_board()
show_board()
print()
print("Saved to:", BOARD_PATH)

## Cell 5: A read-only tool with an optional filter: `list_tasks`

The first tool lets the agent read the board. It returns one line per task.

**The optional `column` filter.** The simple schema form treats every key you list as required. To make `column` optional, the SDK docs suggest a pattern:

1. Leave `column` **out** of the schema. Here the schema is just `{}`.
2. Mention it in the **description**, so Claude knows it can send it.
3. Read it with **`args.get("column")`**, which returns `None` when Claude leaves it out.

With no filter, the tool lists every task. With a filter, it lists only that column.

**`readOnlyHint=True`.** This annotation tells Claude the tool changes nothing. Read-only tools are allowed to run in parallel with other read-only tools. You will test that in Cell 13. Annotations are hints, not enforcement, so keep them honest: this handler really does only read.

| Argument | Value | Why |
|---|---|---|
| `name` | `"list_tasks"` | The agent sees it as `mcp__board__list_tasks` |
| `description` | Explains the tool and the optional `column` | Claude reads this to decide when and how to call it |
| `input_schema` | `{}` | No required arguments |
| `annotations` | `ToolAnnotations(readOnlyHint=True)` | Marks the tool as safe to run side by side |

The tool reads `BOARD` directly. Custom tools run inside this notebook's Python process, so they can see and change your notebook's objects.

In [ ]:
from claude_agent_sdk import tool, ToolAnnotations


@tool(
    "list_tasks",
    "List the tasks on Pixel Forge's sprint board, one line per task. "
    "Optionally pass 'column' (todo, doing or done) to list only that column.",
    {},
    annotations=ToolAnnotations(readOnlyHint=True),
)
async def list_tool(args):
    column = args.get("column")
    lines = []
    for task_id, task in BOARD.items():
        if column and task["column"] != column:
            continue
        lines.append(f"{task_id} | {task['title']} | {task['column']} | {task['owner']}")
    if not lines:
        lines.append(f"No tasks in column '{column}'.")
    return {"content": [{"type": "text", "text": "\n".join(lines)}]}


print("Defined tool:", list_tool.name)

## Cell 6: A tool that changes data: `add_task`

The second tool adds a task. It is the first tool in this course that **changes your application's state**.

**Why full JSON Schema?** Only Maya and Arjun can own a task. The simple schema form cannot express "one of these values", so this tool uses a full JSON Schema dictionary with an `enum`:

| Schema part | Meaning |
|---|---|
| `"type": "object"` | The arguments arrive as one object |
| `"properties"` | Each argument, its type, and a short description Claude can read |
| `"enum": ["Maya", "Arjun"]` | `owner` must be exactly one of these two values |
| `"required"` | Both `title` and `owner` must be sent |

**What the handler does:**

1. Works out the next ID. The highest existing number is 8, so the first new task is `T-9`.
2. Adds the task to `BOARD` in the `todo` column.
3. Calls `save_board()`, so `board.json` matches `BOARD`.
4. Returns the new ID, so the agent can refer to it.

This tool has no `readOnlyHint`. It writes, so it should not be marked as safe to run side by side.

In [ ]:
@tool(
    "add_task",
    "Add a new task to Pixel Forge's sprint board. "
    "New tasks start in the todo column. Returns the new task ID.",
    {
        "type": "object",
        "properties": {
            "title": {"type": "string", "description": "Short task title"},
            "owner": {
                "type": "string",
                "enum": ["Maya", "Arjun"],
                "description": "Who owns the task",
            },
        },
        "required": ["title", "owner"],
    },
)
async def add_tool(args):
    next_number = max((int(task_id.split("-")[1]) for task_id in BOARD), default=0) + 1
    task_id = f"T-{next_number}"
    BOARD[task_id] = {"title": args["title"], "column": "todo", "owner": args["owner"]}
    save_board()
    return {
        "content": [
            {
                "type": "text",
                "text": f"Added {task_id}: {args['title']} ({args['owner']}), in todo.",
            }
        ]
    }


print("Defined tool:", add_tool.name)

## Cell 7: Move a task, with an enum and a clear error: `move_task`

The third tool moves a task between columns. It combines both ideas you have seen so far, and adds one more.

**Full JSON Schema with an enum.** A column can only be `todo`, `doing` or `done`. The `enum` says so, and `required` says the agent must send both `task_id` and `column`. If Claude sends anything else, such as a column called `shipped`, the SDK rejects the call **before your function runs**. You will see that in Cell 11.

**A clear error for an unknown ID.** The schema can check that `task_id` is a string, but it cannot know which IDs exist. Only your code knows that. So the handler checks, and for an unknown ID it returns:

```python
{"content": [...], "is_error": True}
```

The message names the bad ID and lists the IDs that do exist. `is_error` tells Claude the call failed. The list gives it what it needs to explain the problem or retry correctly instead of guessing. You will see that in Cell 12.

**Otherwise** the handler updates `BOARD`, saves to disk, and confirms the move.

In [ ]:
# LIVE CODE - type during recording

## Cell 8: One server, one approval, and a run helper

This cell connects the three tools to the agent.

**One server for all three tools.** `create_sdk_mcp_server()` bundles `list_tool`, `add_tool` and `move_tool` into one in-process MCP server named `board`.

**The options.** `OPTIONS` is a plain dictionary, so every later cell builds its options the same way:

| Option | Value | Why |
|---|---|---|
| `mcp_servers` | `{"board": board_server}` | The key `board` becomes the middle of each tool name: `mcp__board__list_tasks` |
| `allowed_tools` | `["mcp__board__*"]` | The wildcard approves every tool on the `board` server and nothing else. No built-in tools are needed. |
| `model` | `MODEL_NAME` | From Cell 3 |
| `cwd` | `"/content"` | The working directory for the agent in Colab |

The server key and the server `name` are the same string, `board`, to keep things easy to read.

**The `run(prompt)` helper.** Every remaining cell asks the agent something, so this helper does the repetitive part. It runs `query()`, keeps every message, and prints as it goes:

| Printed line | Comes from | Meaning |
|---|---|---|
| `[call]` | `ToolUseBlock` in an `AssistantMessage` | The tool Claude asked for, and its input |
| `[result]` | `ToolResultBlock` in a `UserMessage` | What the tool returned, and whether it was an error |
| `Agent:` | `ResultMessage` | The agent's final answer |

`result_text()` turns a tool result's content into plain text, because the content can arrive as a string or as a list of blocks.

Why print every call? The final answer is only the agent's summary. The stream of calls and results is the evidence of what it actually did. You may also see a `ToolSearch` call, which is Claude loading a tool's full schema on demand.

`run()` is an `async` function, so later cells call it with `await run(...)` at the top level of the cell.

In [ ]:
# LIVE CODE - type during recording

## Cell 9: Read: what is in progress?

The first question only needs to **read** the board. Expect a call to `mcp__board__list_tasks`, possibly with `{"column": "doing"}`. Whether Claude uses the filter or lists everything and picks out the `doing` tasks itself is up to the agent.

This cell also defines two small helpers that the remaining cells reuse:

| Helper | Returns |
|---|---|
| `final_answer(messages)` | The text of the `ResultMessage` |
| `tool_results(messages)` | Every `ToolResultBlock` in the run |

**The check.** The cell prints the `doing` tasks straight from `BOARD` and reports whether each one appears in the agent's answer, by ID or by title. The answer's wording will differ from run to run. The three tasks and their owners should not.

In [ ]:
def final_answer(messages):
    for message in messages:
        if isinstance(message, ResultMessage):
            return message.result or ""
    return ""


def tool_results(messages):
    results = []
    for message in messages:
        if isinstance(message, UserMessage) and isinstance(message.content, list):
            results += [b for b in message.content if isinstance(b, ToolResultBlock)]
    return results


messages = await run("What tasks are in progress right now, and who owns each one?")

answer = final_answer(messages).lower()
print()
print("Ground truth from BOARD (doing):")
for task_id, task in BOARD.items():
    if task["column"] == "doing":
        mentioned = task_id.lower() in answer or task["title"].lower() in answer
        print(f"  {task_id}  {task['title']}  ({task['owner']})  in answer: {mentioned}")

## Cell 10: Change: add a task and move another

Now the agent **changes** the board. One prompt asks for two changes:

1. Add "Tune boss fight difficulty" for Maya. Expect a call to `add_task`.
2. Move T-3 to `done`. Expect a call to `move_task`.

The goal prompt does not say which tools to use or in what order. The agent decides.

**The proof is on disk.** The cell reads `board.json` before and after the run, prints both, and then checks two facts in the file:

| Check | Expected |
|---|---|
| A task titled "Tune boss fight difficulty" | In `todo`, owned by Maya, with ID `T-9` if this is the first run since Cell 4 |
| T-3 | In `done` |

If you run this cell twice without rerunning Cell 4, a second copy of the task is added. That is the tool doing exactly what it was asked.

In [ ]:
def print_file(board):
    for task_id, task in board.items():
        print(f"  {task_id}  {task['column']:<6} {task['owner']:<6} {task['title']}")


before = read_board_file()
print("board.json BEFORE:")
print_file(before)
print()

messages = await run(
    "Add a task called 'Tune boss fight difficulty' for Maya, then move T-3 to done."
)

after = read_board_file()
print()
print("board.json AFTER:")
print_file(after)

new_ids = [task_id for task_id in after if task_id not in before]
new_task_ok = any(
    after[task_id]["title"].strip().lower() == "tune boss fight difficulty"
    and after[task_id]["column"] == "todo"
    and after[task_id]["owner"] == "Maya"
    for task_id in new_ids
)
print()
print("New task IDs on disk:", new_ids)
print("New task in todo, owned by Maya:", new_task_ok)
print("T-3 is in done:", after["T-3"]["column"] == "done")

## Cell 11: Invalid input: a column that does not exist

What happens if the agent sends a value the schema does not allow? This prompt asks for T-2 to move to a column called `shipped`, and tells Claude to use exactly that name.

`move_task`'s schema only allows `todo`, `doing` and `done`. The SDK checks every call against the tool's schema **before** your function runs. A call with `"column": "shipped"` is rejected, and Claude gets an error result that starts with `Input validation error:`. The handler never runs, so it never touches `BOARD` or `board.json`.

**What the cell prints:**

- Every tool result with `is_error` set, and its text
- Whether `board.json` is unchanged
- Which column T-2 is in now
- The agent's reply, from `run()`

**Two outcomes are possible, and both are good:**

| What you see | What it means |
|---|---|
| An `is_error` result with `Input validation error` | Claude tried `shipped`. The schema stopped it before your code ran. |
| No tool call for `move_task` at all | Claude read the allowed values from the schema and refused up front. That is also the schema doing its job. |

Either way, `board.json` should be unchanged. Your function only ever receives arguments that match the schema.

In [ ]:
before = BOARD_PATH.read_text()

messages = await run("Move T-2 to the column called 'shipped'. Use exactly that column name.")

print()
errors = [block for block in tool_results(messages) if block.is_error]
for block in errors:
    print("is_error result:", result_text(block))
if not errors:
    print("No error result: the agent did not send 'shipped' to the tool.")

print("board.json unchanged:", BOARD_PATH.read_text() == before)
print("T-2 is in:", read_board_file()["T-2"]["column"])

## Cell 12: Error result: a task that does not exist

This time the input is perfectly valid. `T-99` is a string and `done` is an allowed column, so the schema lets the call through and **your handler runs**.

The handler looks up `T-99`, does not find it, and takes the error branch you wrote in Cell 7. It returns `is_error: True` with a message that names the bad ID and lists the IDs that exist.

**Compare the two kinds of error:**

| | Cell 11 | Cell 12 |
|---|---|---|
| Caught by | The schema, before your function runs | Your own code |
| Message written by | The SDK | You |
| What Claude learns | Which values are allowed | That T-99 does not exist, and which IDs do |

Because the message is clear, the agent can explain the problem to the user instead of guessing. Watch its reply. It should say the task does not exist, and it may list the real IDs.

The cell then confirms that `board.json` is unchanged.

In [ ]:
before = BOARD_PATH.read_text()

messages = await run("Move task T-99 to done.")

print()
errors = [block for block in tool_results(messages) if block.is_error]
for block in errors:
    print("is_error result:", result_text(block))
if not errors:
    print("No error result in this run.")

print("board.json unchanged:", BOARD_PATH.read_text() == before)

## Cell 13: Parallel read-only calls

`list_tasks` carries `readOnlyHint=True`. That hint tells Claude the tool changes nothing, which makes it safe to run several calls side by side.

This prompt asks for two separate lists: everything in `todo` and everything in `done`. A natural plan is two `list_tasks` calls, one per column. Because the tool is read-only, Claude may send both calls in the **same** response instead of waiting for the first result before asking for the second.

**How the cell checks.** Several `ToolUseBlock`s in one model response mean parallel calls in one turn. The cell counts the `list_tasks` calls in each `AssistantMessage`. Claude Code can also deliver the blocks of one response as separate `AssistantMessage`s, so the cell also compares each message's `message_id`, which identifies the model response the blocks came from.

**Be ready for any of these:**

| What you see | What it means |
|---|---|
| Two calls in the same response | Parallel calls. The read-only hint allowed it. |
| Two calls in different responses | One after another. `readOnlyHint` permits parallel calls, it does not force them. |
| One call with no filter | Claude listed everything once and split the lists itself. Also a valid plan. |

`readOnlyHint` is a permission for safe parallelism, not an instruction.

In [ ]:
LIST_TOOL = "mcp__board__list_tasks"

messages = await run("Give me two separate lists: everything in todo and everything in done. Use list_tasks with its column filter for each list.")

per_message = []
for message in messages:
    if isinstance(message, AssistantMessage):
        calls = [
            block.input
            for block in message.content
            if isinstance(block, ToolUseBlock) and block.name == LIST_TOOL
        ]
        if calls:
            per_message.append((message.message_id, calls))

print()
for number, (message_id, calls) in enumerate(per_message, 1):
    print(f"AssistantMessage {number}: {len(calls)} list_tasks call(s) {calls}  message_id={message_id}")

total = sum(len(calls) for _, calls in per_message)
ids = [message_id for message_id, _ in per_message]
same_message = any(len(calls) >= 2 for _, calls in per_message)
same_response = same_message or (total >= 2 and None not in ids and len(set(ids)) == 1)

print()
print("Total list_tasks calls:", total)
print("Two or more calls in one AssistantMessage:", same_message)
print("Two or more calls in one model response:", same_response)